# Suppressing instability on a Vlasov-Poisson system - Bump-on-Tail (NVIDIA Warp backend)

Warp port of the Bump-on-Tail example.  Same custom_vjp adjoint, different equilibrium distribution and cost function.


## Setup


In [ ]:
# Colab setup - skip if running locally with the package already installed.
import importlib, subprocess, sys

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or pkg,
        ])

for pkg, pip_name in [
    ('warp', 'warp-lang'),
    ('jax', 'jax'),
    ('optax', 'optax'),
    ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'),
]:
    _ensure(pkg, pip_name)

# Install the warp_vp_solver package itself.
try:
    import warp_vp_solver  # noqa: F401
except ImportError:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/Forgotten/warp_vp_solver.git@main',
    ])

import warp as wp
wp.init()
print(f'Warp device: {wp.get_preferred_device()}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import optax
from tqdm import trange

from warp_vp_solver import (
    make_mesh,
    WarpVlasovPoissonSolver,
    external_electric_field,
    make_cost_function_kl,
    make_cost_function_ee,
    plot_inital_solve,
    plot_results_TS,
    plot_results_BoT,
)

jax.config.update('jax_enable_x64', True)


## Bump-on-Tail equilibrium and initial condition


In [ ]:
nx, nv = 128, 128
L = 20.0 * np.pi
LV = 8.0
dt = 0.1
t_final = 40.0
t_values = np.linspace(0.0, t_final, int(t_final / dt))
k_0 = 0.1

mesh = make_mesh(L, LV, nx, nv)

alpha = 0.9
vt2 = 0.5
vd = 4.5
f_main = alpha * np.exp(-0.5 * mesh.V ** 2) / np.sqrt(2.0 * np.pi)
f_bump = ((1.0 - alpha)
          * np.exp(-0.5 * (mesh.V - vd) ** 2 / vt2)
          / np.sqrt(2.0 * np.pi * vt2))
f_eq = f_main + f_bump

epsilon = 0.04
f_iv = (1.0 + epsilon * np.cos(k_0 * mesh.X)) * f_eq


## Forward solver with H=0


In [ ]:
solver = WarpVlasovPoissonSolver(mesh=mesh, dt=dt, f_eq=f_eq)
solver_jit = solver.run_forward_jax

H_zero = jnp.zeros(mesh.nx)
f0_out, fh0, Eh0, ee0 = solver_jit(jnp.asarray(f_iv), H_zero, t_final)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_values, np.asarray(ee0))
ax.set_xlabel('$t$')
ax.set_ylabel('Electric energy')
ax.set_title('Bump-on-Tail (H=0)')
plt.show()


## Minimize final-time electric energy via Optax


In [ ]:
k_total = 2
ak_init = jax.random.uniform(
    jax.random.key(123), (2, k_total), minval=-0.005, maxval=0.005,
)

cost_fn_ee = make_cost_function_ee(
    solver, solver_jit, jnp.asarray(f_iv), k_0, t_final,
)
value_and_grad = jax.value_and_grad(cost_fn_ee)

opt = optax.adam(learning_rate=5e-4)
params = ak_init
opt_state = opt.init(params)

history = []
for step in trange(20):
    val, grad = value_and_grad(params)
    updates, opt_state = opt.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    history.append(float(val))

history = np.asarray(history)
print('Final E-energy:', history[-1])


## Final solution and convergence


In [ ]:
H_opt = external_electric_field(params, mesh, k_0)
f_opt, fh_opt, Eh_opt, ee_opt = solver_jit(
    jnp.asarray(f_iv), H_opt, t_final,
)

fig, axs = plt.subplots(1, 4, figsize=(28, 6))
plot_results_BoT(
    fig, axs, f_opt, Eh_opt, np.asarray(H_opt), ee_opt,
    history, t_values, mesh,
)
plt.show()
